# 07_Data_Preprocessing.ipynb

In [ ]:
!pip -q install pandas scikit-learn

import pandas as pd
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from google.colab import files

print('Upload cnn_features.csv')
u=files.upload()
cnn=pd.read_csv(list(u.keys())[0])

print('Upload era5_features.csv')
u=files.upload()
era5=pd.read_csv(list(u.keys())[0])

print('Upload modis_features.csv')
u=files.upload()
modis=pd.read_csv(list(u.keys())[0])

for df in [cnn,era5,modis]:
    df['event_id']=df['event_id'].astype(str)

era5=era5.drop(columns=['image_date'],errors='ignore')
modis=modis.drop(columns=['image_date'],errors='ignore')

merged=cnn.merge(era5,on='event_id',how='left')
merged=merged.merge(modis,on='event_id',how='left')
merged=merged.drop_duplicates(subset='event_id')

num=merged.select_dtypes(include='number').columns
imp=SimpleImputer(strategy='median')
merged[num]=imp.fit_transform(merged[num])

weather=[c for c in ['era5_temp_mean_c','era5_dewpoint_mean_c','era5_pressure_mean_pa','era5_u_wind_mean','era5_v_wind_mean','era5_precip_sum_mm','era5_runoff_sum_mm','LST_Day_C','LST_Night_C'] if c in merged.columns]
if weather:
    merged[weather]=StandardScaler().fit_transform(merged[weather])

merged.to_csv('training_dataset.csv',index=False)
print(merged.shape)
files.download('training_dataset.csv')
